the problem for our model at least right now is that its not able to do any reconstruction at all lol.   


What if, instead of MSE, i just take the batch's residuals and subtract `sigma_eps*<noise vector randomly sampled each time>` -> this should be zero. so laplace. so mod of this value for the whole batch is the loss term.   
This might be very weird.   


On top of this, we will have the weights regularisation. rmasked) 

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
MODE = "light"

In [ ]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from neural_data.models import *
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
    show_72,
    show_72_list,
    scatter_plot_1d,
    mk_rect_on_ax,
    get_receptive,
    otsu_threshold,
    explain_variance_with_pca,
)
from tqdm import tqdm
from pt_to_api.contribs.v1 import (
    show_input_patch_and_kernel_placement_for_poi_using_raw_params as SIP,
)
import seaborn as sns
import numpy as np
from sklearn.decomposition import MiniBatchDictionaryLearning, DictionaryLearning
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
import pandas as pd

from neural_data.utils import (
    get_full_conv_kernel_at_coordinate,
    get_saliency_map_ids_and_patches,
    get_full_activations_of_layer,
)
from collections import defaultdict
import itertools
from torch import nn
from torch import optim


In [ ]:
from nilearn import datasets

# Here we use only single subject to get faster-running code.
dataset = datasets.fetch_development_fmri(n_subjects=1)
func_filename = dataset.func[0]

# print basic information on the dataset
print(f"First subject functional nifti image (4D) is at: {dataset.func[0]}")

# fMRI

https://nilearn.github.io/stable/auto_examples/03_connectivity/plot_compare_decomposition.html#sphx-glr-auto-examples-03-connectivity-plot-compare-decomposition-py

In [ ]:
from nilearn.maskers import NiftiMasker

# This is fMRI timeseries data:
# the background has not been removed yet,
# thus we need to use mask_strategy='epi' to compute the mask from the
# EPI images
masker = NiftiMasker(
    smoothing_fwhm=8,
    memory="nilearn_cache",
    memory_level=1,
    mask_strategy="epi",
    verbose=1,
)
data_masked = masker.fit_transform(func_filename)

In [ ]:
data_masked.shape

In [ ]:
def thresholded(components_masked):
    components_masked[np.abs(components_masked) < 0.8] = 0
    return components_masked

    # Now invert the masking operation, going back to a full 3D
    # representation
    # component_img = masker.inverse_transform(components_masked)

In [ ]:
from sklearn.decomposition import FastICA

n_components = 20
ica = FastICA(
    n_components=n_components, random_state=42, max_iter=2000, tol=0.01
)
components_masked = ica.fit_transform(data_masked.T).T
components_masked -= components_masked.mean(axis=0)
components_masked /= components_masked.std(axis=0)

# # Normalize estimated components, for thresholding to make sense

In [ ]:
shape = (155,155)
dims = shape[0] * shape[1]

S([c[:dims].reshape(shape) for c in thresholded(components_masked)], (20,15), 5)
plt.show()

In [ ]:
from pt_to_api.benchmark import train_x as TX

In [ ]:
from pt_to_api import benchmark as B

scaler = B.NormaliseStdScaler().fit(data_masked)
data_scaled = scaler.transform(data_masked)

In [ ]:
np.sqrt(12)

In [ ]:
run = TX.train(data_scaled, 5, 1e-2, 1000, baseline_epochs=400, device="mps")

In [ ]:
B.get_metrics_from_run(run)

In [ ]:
r_comps = scaler.inverse_transform(run.components)

In [ ]:
S([c[:dims].reshape(shape) for c in r_comps], (20,15), 5)
plt.show()

    # if sigma_eps_override is not None:
    #     sigma_eps = sigma_eps_override

In [ ]:
r_comps.shape

In [ ]:
run.components.shape

In [ ]:
plt.imshow(components_masked[1][:24025].reshape(155,155))